# LG-CoTrain: Best-Epoch Phase 1 Seeding — 20 Epochs

This notebook is identical to [Notebook 11](11_best_epoch_seeding_run.ipynb) but runs
Phase 1 for **20 epochs** instead of the default 7, giving the best-epoch selection a
wider search window.

| Setting | Value |
|---|---|
| Phase 1 seed strategy | `best` (seed from best-performing Phase 1 epoch) |
| Phase 1 epochs | **20** (default is 7) |
| Stopping strategy | `baseline` (paper's Algorithm 1: patience on ensemble macro-F1) |
| Hyperparameters | Per-experiment Optuna-tuned (same as Notebook 10) |
| Results path | `results/gpt-4o/test/run-5` |

**Total experiments**: 10 events x 4 budgets x 3 seeds = **120 runs**

**Rationale**: With `phase1_seed_strategy="best"`, Phase 1 overfitting is harmless —
overfit epochs are simply not selected. More epochs = more chances to find the optimal
seeding point where the models have learned useful patterns but have not yet memorized
the tiny labeled splits. The cost is minimal since Phase 1 trains on D_l1/D_l2 (very
small splits), so 20 epochs adds only ~15-20 seconds of GPU time per experiment.

**Comparison baselines**:
- Notebook 11: `best` seeding, 7 epochs (`results/gpt-4o/test/run-4`)
- Notebook 10 baseline: `last` seeding, 7 epochs (`results/gpt-4o/optuna-stop/baseline/`)

## Resume Support

Every `(event, budget, seed_set)` combination is written to its own `metrics.json`
as soon as it completes. Re-running any cell automatically skips completed experiments.

In [ ]:
import json
import statistics
import sys
import time
from pathlib import Path


def _find_repo_root(marker: str = "lg_cotrain") -> Path:
    for candidate in [Path().resolve()] + list(Path().resolve().parents):
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(
        f"Cannot find repo root: no ancestor directory contains '{marker}/'. "
        "Run the notebook from inside the repository."
    )


repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import numpy as np

from lg_cotrain.run_all import BUDGETS, SEED_SETS, format_summary_table, run_all_experiments
from lg_cotrain.parallel import run_experiments_parallel
from lg_cotrain.optuna_per_experiment import load_best_params


class ProgressTracker:
    """Track global progress across all events x budgets x seeds."""

    def __init__(self, total: int, already_done: int, start_time: float):
        self.total = total
        self.done = already_done
        self.start_time = start_time

    def update(self, event, budget, seed_set, status):
        self.done += 1
        elapsed = time.time() - self.start_time
        pct = 100.0 * self.done / self.total if self.total else 0
        elapsed_h = elapsed / 3600
        remaining = self.total - self.done
        eta_h = (elapsed / self.done) * remaining / 3600 if self.done > 0 else 0
        print(
            f"  [PROGRESS] {self.done}/{self.total} ({pct:.1f}%)"
            f"  |  Elapsed: {elapsed_h:.2f}h  |  ETA: {eta_h:.2f}h  |  {status}"
        )


print(f"Repo root : {repo_root}")
print(f"Budgets   : {BUDGETS}")
print(f"Seed sets : {SEED_SETS}")

In [ ]:
# ---- Configuration ----

PSEUDO_LABEL_SOURCE = "gpt-4o"

# Number of GPUs for parallel execution (1 = sequential fallback)
NUM_GPUS = 2

# Optuna trial count to load (None = latest available)
N_TRIALS = 10

# Phase 1 seed strategy: "best" = seed from best Phase 1 epoch (our extension)
PHASE1_SEED_STRATEGY = "best"

# Phase 1 epochs: 20 instead of default 7 for wider best-epoch search window
WEIGHT_GEN_EPOCHS = 20

# Only baseline stopping strategy (paper's Algorithm 1)
STOPPING_STRATEGY = "baseline"

DATA_ROOT = str(repo_root / "data")

# Results destination
RESULTS_ROOT = str(repo_root / "results" / "gpt-4o" / "test" / "run-5")

# Discover all events
TARGET_EVENTS = sorted(
    p.name for p in (Path(DATA_ROOT) / "original").iterdir() if p.is_dir()
)

# ---- Load Optuna best params for ALL (event, budget, seed_set) combos ----
OPTUNA_DIR = str(repo_root / "results" / "optuna" / "per_experiment")
optuna_results = load_best_params(
    storage_dir=OPTUNA_DIR,
    n_trials=N_TRIALS,
)

# Default hyperparams (fallback when Optuna results are missing)
DEFAULT_PARAMS = {
    "lr": 2e-5,
    "batch_size": 32,
    "cotrain_epochs": 10,
    "finetune_patience": 5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
}

# Build per-experiment param lookup: (event, budget, seed_set) -> params dict
experiment_params = {}  # key -> 6-hyperparam dict
missing_keys = []
for event in TARGET_EVENTS:
    for budget in BUDGETS:
        for seed_set in SEED_SETS:
            key = (event, budget, seed_set)
            if key in optuna_results:
                experiment_params[key] = optuna_results[key]["best_params"]
            else:
                experiment_params[key] = None  # will use defaults
                missing_keys.append(key)

# ---- Print summary ----
total_experiments = len(TARGET_EVENTS) * len(BUDGETS) * len(SEED_SETS)
n_with_optuna = sum(1 for v in experiment_params.values() if v is not None)
n_with_default = sum(1 for v in experiment_params.values() if v is None)

print(f"Phase 1 seed  : {PHASE1_SEED_STRATEGY}")
print(f"Phase 1 epochs: {WEIGHT_GEN_EPOCHS}")
print(f"Stopping      : {STOPPING_STRATEGY}")
print(f"Events        : {len(TARGET_EVENTS)} events")
print(f"Total expts   : {total_experiments} experiments  ({len(TARGET_EVENTS)} events x {len(BUDGETS)} budgets x {len(SEED_SETS)} seeds)")
print(f"GPUs          : {NUM_GPUS}")
print(f"Optuna params : {n_with_optuna}/{total_experiments} loaded  |  {n_with_default} using defaults")
print(f"Optuna dir    : {OPTUNA_DIR}  (n_trials={N_TRIALS})")
print(f"Results root  : {RESULTS_ROOT}")

if missing_keys:
    print(f"\nWARNING: {len(missing_keys)} experiment(s) missing Optuna results, using defaults:")
    for k in missing_keys[:10]:
        print(f"  {k}")
    if len(missing_keys) > 10:
        print(f"  ... and {len(missing_keys) - 10} more")

## Running Experiments

The cell below runs all 120 experiments (baseline stopping + best-epoch seeding + 20 Phase 1 epochs)
with per-experiment Optuna hyperparameters. Completed experiments are skipped automatically.

> **Runtime estimate**: With 2 GPUs, ~2-3 hours total (slightly longer than NB11 due to 20 vs 7 Phase 1 epochs).
> Run overnight or split across sessions -- resume support ensures no work is lost.

In [ ]:
print(f"Total experiments : {total_experiments}")
print(f"Strategy          : {STOPPING_STRATEGY}")
print(f"Phase 1 seeding   : {PHASE1_SEED_STRATEGY}")
print(f"Phase 1 epochs    : {WEIGHT_GEN_EPOCHS}")
print(f"Execution mode    : {'parallel (' + str(NUM_GPUS) + ' GPUs)' if NUM_GPUS > 1 else 'sequential'}")
print()

overall_start = time.time()
tracker = ProgressTracker(total_experiments, 0, overall_start)
event_results = {}  # event -> list[result_dict]

print(f"{'=' * 70}")
print(f"Strategy: {STOPPING_STRATEGY}  |  Phase 1 seeding: {PHASE1_SEED_STRATEGY}  |  Phase 1 epochs: {WEIGHT_GEN_EPOCHS}")
print(f"Results -> {RESULTS_ROOT}")
print(f"{'=' * 70}")

# Build list of pending experiment configs, skipping completed ones
pending_configs = []   # list of LGCoTrainConfig kwarg dicts
pending_keys = []      # corresponding (event, budget, seed_set) tuples
skipped_results = {}   # (event, budget, seed_set) -> result dict

for event in TARGET_EVENTS:
    for budget in BUDGETS:
        for seed_set in SEED_SETS:
            metrics_path = (
                Path(RESULTS_ROOT) / event / f"{budget}_set{seed_set}" / "metrics.json"
            )

            if metrics_path.exists():
                with open(metrics_path) as f:
                    skipped_results[(event, budget, seed_set)] = json.load(f)
                tracker.update(event, budget, seed_set, "skipped")
                continue

            # Per-experiment Optuna hyperparams (or defaults)
            p = experiment_params.get((event, budget, seed_set)) or DEFAULT_PARAMS
            pending_configs.append(dict(
                event=event,
                budget=budget,
                seed_set=seed_set,
                pseudo_label_source=PSEUDO_LABEL_SOURCE,
                stopping_strategy=STOPPING_STRATEGY,
                phase1_seed_strategy=PHASE1_SEED_STRATEGY,
                weight_gen_epochs=WEIGHT_GEN_EPOCHS,
                lr=p["lr"],
                batch_size=p["batch_size"],
                cotrain_epochs=p["cotrain_epochs"],
                finetune_patience=p["finetune_patience"],
                weight_decay=p["weight_decay"],
                warmup_ratio=p["warmup_ratio"],
                data_root=DATA_ROOT,
                results_root=RESULTS_ROOT,
            ))
            pending_keys.append((event, budget, seed_set))

n_skipped = len(skipped_results)
n_pending = len(pending_configs)
print(f"  Skipped (completed): {n_skipped}  |  Pending: {n_pending}")

# Dispatch pending experiments
parallel_results = {}  # (event, budget, seed_set) -> result dict

if pending_configs and NUM_GPUS > 1:
    print(f"  Dispatching {n_pending} experiments across {NUM_GPUS} GPUs...")
    outcomes = run_experiments_parallel(
        pending_configs,
        num_gpus=NUM_GPUS,
        on_experiment_done=tracker.update,
    )
    for key, outcome in zip(pending_keys, outcomes):
        if outcome["status"] == "done" and outcome["result"] is not None:
            parallel_results[key] = outcome["result"]
        else:
            print(f"  WARNING: {key} failed")

elif pending_configs:
    # Sequential fallback (NUM_GPUS == 1)
    for key, cfg in zip(pending_keys, pending_configs):
        results = run_all_experiments(
            key[0],
            budgets=[key[1]],
            seed_sets=[key[2]],
            pseudo_label_source=PSEUDO_LABEL_SOURCE,
            stopping_strategy=STOPPING_STRATEGY,
            phase1_seed_strategy=PHASE1_SEED_STRATEGY,
            weight_gen_epochs=WEIGHT_GEN_EPOCHS,
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            cotrain_epochs=cfg["cotrain_epochs"],
            finetune_patience=cfg["finetune_patience"],
            weight_decay=cfg["weight_decay"],
            warmup_ratio=cfg["warmup_ratio"],
            data_root=DATA_ROOT,
            results_root=RESULTS_ROOT,
            _on_experiment_done=tracker.update,
        )
        if results and results[0] is not None:
            parallel_results[key] = results[0]

# Merge all results into per-event lists
all_results_flat = {**skipped_results, **parallel_results}
for event in TARGET_EVENTS:
    evt_results = [
        all_results_flat.get((event, b, s))
        for b in BUDGETS
        for s in SEED_SETS
    ]
    evt_results = [r for r in evt_results if r is not None]
    if evt_results:
        event_results[event] = evt_results
        print(f"\n  {format_summary_table(evt_results, event)}")

total_elapsed = time.time() - overall_start
print(f"\n{'=' * 70}")
print(f"All experiments complete in {total_elapsed / 3600:.2f}h ({total_elapsed / 60:.1f}min)")

In [ ]:
# Load any results that already existed (re-run safe)
for event in TARGET_EVENTS:
    if event in event_results:
        continue
    results = []
    for budget in BUDGETS:
        for seed_set in SEED_SETS:
            path = Path(RESULTS_ROOT) / event / f"{budget}_set{seed_set}" / "metrics.json"
            if path.exists():
                with open(path) as f:
                    results.append(json.load(f))
    if results:
        event_results[event] = results

# Build lookup: lookup[event][(budget, seed_set)] -> result
lookup = {}
for event in TARGET_EVENTS:
    lookup[event] = {}
    for r in event_results.get(event, []):
        if r is not None:
            lookup[event][(r["budget"], r["seed_set"])] = r

# Coverage report
n_loaded = sum(len(lookup[e]) for e in TARGET_EVENTS)
print(f"Experiments loaded: {n_loaded}/{total_experiments}  ({100 * n_loaded / total_experiments:.0f}%)")

In [ ]:
# Per-event summary tables: macro-F1 per budget (mean +/- std across 3 seeds)

col_w = 18

for event in TARGET_EVENTS:
    print(f"\n{'=' * 70}")
    print(f"Event: {event}")
    print(f"{'=' * 70}")

    print(f"{'Budget':<8}" + "".join(f" {'Macro-F1':<{col_w}}" for _ in ['']) + "  Error Rate")
    print("-" * 50)

    for budget in BUDGETS:
        f1s = [
            lookup[event].get((budget, s), {}).get("test_macro_f1")
            for s in SEED_SETS
        ]
        errs = [
            lookup[event].get((budget, s), {}).get("test_error_rate")
            for s in SEED_SETS
        ]
        f1s = [f for f in f1s if f is not None]
        errs = [e for e in errs if e is not None]

        if f1s:
            f_m = statistics.mean(f1s)
            f_sd = statistics.stdev(f1s) if len(f1s) >= 2 else 0.0
            e_m = statistics.mean(errs) if errs else 0
            print(f"{budget:<8} {f_m:.4f}+/-{f_sd:.4f}        {e_m:.2f}%")
        else:
            print(f"{budget:<8} N/A")

In [ ]:
# Bar chart: macro-F1 by budget, one subplot per event

n_events = len(TARGET_EVENTS)
ncols = min(5, n_events)
nrows = (n_events + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4.5 * nrows), sharey=False)
axes_flat = np.array(axes).flatten() if n_events > 1 else [axes]

for ax, event in zip(axes_flat, TARGET_EVENTS):
    x = np.arange(len(BUDGETS))
    means, errs = [], []
    for budget in BUDGETS:
        f1s = [
            lookup[event].get((budget, s), {}).get("test_macro_f1")
            for s in SEED_SETS
        ]
        f1s = [f for f in f1s if f is not None]
        means.append(statistics.mean(f1s) if f1s else 0)
        errs.append(statistics.stdev(f1s) if len(f1s) >= 2 else 0)

    ax.bar(x, means, 0.6, yerr=errs, capsize=3, color="darkorange", alpha=0.85)
    ax.set_title(event.replace("_", " ").title(), fontsize=9)
    ax.set_xlabel("Budget")
    ax.set_ylabel("Macro-F1")
    ax.set_xticks(x)
    ax.set_xticklabels([str(b) for b in BUDGETS])
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.3)

# Hide unused subplots
for ax in axes_flat[n_events:]:
    ax.set_visible(False)

fig.suptitle(
    f"Best-Epoch Phase 1 Seeding, {WEIGHT_GEN_EPOCHS} Epochs (Optuna Hyperparams) -- Baseline Stopping\n"
    f"(pseudo-labels: {PSEUDO_LABEL_SOURCE}, phase1_seed_strategy={PHASE1_SEED_STRATEGY})",
    fontsize=12,
)
plt.tight_layout()
plt.show()

In [ ]:
# Cross-event grand summary: mean macro-F1 across all events per budget

print(f"Grand summary -- mean macro-F1 across all events and seeds")
print(f"  Phase 1 seeding: {PHASE1_SEED_STRATEGY}  |  Phase 1 epochs: {WEIGHT_GEN_EPOCHS}  |  Stopping: {STOPPING_STRATEGY}")
print()
print(f"{'Budget':<8}" + "  Mean Macro-F1")
print("-" * 35)

all_f1s_total = []
for budget in BUDGETS:
    f1s = [
        lookup[e].get((budget, s), {}).get("test_macro_f1")
        for e in TARGET_EVENTS
        for s in SEED_SETS
    ]
    f1s = [f for f in f1s if f is not None]
    if f1s:
        m = statistics.mean(f1s)
        sd = statistics.stdev(f1s) if len(f1s) >= 2 else 0
        all_f1s_total.extend(f1s)
        print(f"{budget:<8}  {m:.4f} +/- {sd:.4f}  (n={len(f1s)})")
    else:
        print(f"{budget:<8}  N/A")

if all_f1s_total:
    overall = statistics.mean(all_f1s_total)
    print(f"{'Overall':<8}  {overall:.4f}  (n={len(all_f1s_total)})")

## Comparison: 3-Way (last/7ep vs best/7ep vs best/20ep)

Compare this notebook's results against:
- **NB10 baseline**: `last` seeding, 7 Phase 1 epochs (paper's default)
- **NB11**: `best` seeding, 7 Phase 1 epochs
- **NB12 (this)**: `best` seeding, 20 Phase 1 epochs

In [ ]:
# Load comparison baselines
LAST_7EP_ROOT = str(repo_root / "results" / PSEUDO_LABEL_SOURCE / "optuna-stop" / "baseline")  # NB10
BEST_7EP_ROOT = str(repo_root / "results" / "gpt-4o" / "test" / "run-4")  # NB11

def _load_lookup(root):
    lkp = {}
    for event in TARGET_EVENTS:
        lkp[event] = {}
        for budget in BUDGETS:
            for seed_set in SEED_SETS:
                path = Path(root) / event / f"{budget}_set{seed_set}" / "metrics.json"
                if path.exists():
                    with open(path) as f:
                        lkp[event][(budget, seed_set)] = json.load(f)
    return lkp

last_7ep_lookup = _load_lookup(LAST_7EP_ROOT)
best_7ep_lookup = _load_lookup(BEST_7EP_ROOT)

n_last_7 = sum(len(last_7ep_lookup[e]) for e in TARGET_EVENTS)
n_best_7 = sum(len(best_7ep_lookup[e]) for e in TARGET_EVENTS)
print(f"Loaded {n_last_7} 'last/7ep' results (NB10) from: {LAST_7EP_ROOT}")
print(f"Loaded {n_best_7} 'best/7ep' results (NB11) from: {BEST_7EP_ROOT}")
print()

# ---- Per-event comparison ----
configs = [
    ("last/7ep (NB10)", last_7ep_lookup),
    ("best/7ep (NB11)", best_7ep_lookup),
    ("best/20ep (NB12)", lookup),
]

print("3-Way Comparison: Mean Test Macro-F1")
print("=" * 100)

for event in TARGET_EVENTS:
    print(f"\nEvent: {event}")
    header = f"{'Budget':<8}"
    for label, _ in configs:
        header += f" {label:>16}"
    header += f" {'Delta 12-10':>12}"
    print(header)
    print("-" * len(header))

    for budget in BUDGETS:
        row = f"{budget:<8}"
        means = []
        for label, lkp in configs:
            f1s = [
                lkp[event].get((budget, s), {}).get("test_macro_f1")
                for s in SEED_SETS
            ]
            f1s = [f for f in f1s if f is not None]
            m = statistics.mean(f1s) if f1s else None
            means.append(m)
            row += f" {m:.4f}" if m is not None else f" {'N/A':>16}"

        # Delta: best/20ep vs last/7ep
        if means[0] is not None and means[2] is not None:
            d = means[2] - means[0]
            sign = "+" if d >= 0 else ""
            row += f" {sign}{d:>11.4f}"
        else:
            row += f" {'N/A':>12}"
        print(row)

# ---- Grand summary ----
print(f"\n{'=' * 100}")
print(f"Grand Summary (all events, all seeds)")
header = f"{'Budget':<8}"
for label, _ in configs:
    header += f" {label:>16}"
header += f" {'Delta 12-10':>12}"
print(header)
print("-" * len(header))

grand_means = {label: [] for label, _ in configs}

for budget in BUDGETS:
    row = f"{budget:<8}"
    budget_means = []
    for label, lkp in configs:
        f1s = [
            lkp[e].get((budget, s), {}).get("test_macro_f1")
            for e in TARGET_EVENTS for s in SEED_SETS
        ]
        f1s = [f for f in f1s if f is not None]
        m = statistics.mean(f1s) if f1s else None
        budget_means.append(m)
        if f1s:
            grand_means[label].extend(f1s)
        row += f" {m:.4f}" if m is not None else f" {'N/A':>16}"

    if budget_means[0] is not None and budget_means[2] is not None:
        d = budget_means[2] - budget_means[0]
        sign = "+" if d >= 0 else ""
        row += f" {sign}{d:>11.4f}"
    else:
        row += f" {'N/A':>12}"
    print(row)

# Overall row
row = f"{'Overall':<8}"
overall_vals = []
for label, _ in configs:
    vals = grand_means[label]
    m = statistics.mean(vals) if vals else None
    overall_vals.append(m)
    row += f" {m:.4f}" if m is not None else f" {'N/A':>16}"
if overall_vals[0] is not None and overall_vals[2] is not None:
    d = overall_vals[2] - overall_vals[0]
    sign = "+" if d >= 0 else ""
    row += f" {sign}{d:>11.4f}"
print(row)

# ---- 3-way bar chart per budget ----
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(BUDGETS))
n_bars = len(configs)
w = 0.8 / n_bars
colors = ["steelblue", "darkorange", "forestgreen"]

for i, (label, lkp) in enumerate(configs):
    means = []
    for budget in BUDGETS:
        f1s = [lkp[e].get((budget, s), {}).get("test_macro_f1")
               for e in TARGET_EVENTS for s in SEED_SETS]
        f1s = [f for f in f1s if f is not None]
        means.append(statistics.mean(f1s) if f1s else 0)
    offset = (i - n_bars / 2 + 0.5) * w
    ax.bar(x + offset, means, w * 0.9, label=label, color=colors[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([str(b) for b in BUDGETS])
ax.set_xlabel("Budget (labels per class)")
ax.set_ylabel("Mean Test Macro-F1")
ax.set_ylim(0, 1)
ax.set_title(
    f"Phase 1 Seeding: 3-Way Comparison (Optuna Hyperparams, Baseline Stopping)\n"
    f"Mean across all events and seeds (pseudo-labels: {PSEUDO_LABEL_SOURCE})",
    fontsize=11,
)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class F1 heatmaps at budget=5 (hardest case)
# Compare last/7ep vs best/7ep vs best/20ep

from lg_cotrain.data_loading import CLASS_LABELS

HEATMAP_BUDGET = 5
print(f"Per-class F1 heatmaps at budget={HEATMAP_BUDGET}: 3-way comparison")

for event in TARGET_EVENTS:
    data_rows = []
    row_labels = []

    for label, lkp in [("last/7ep (NB10)", last_7ep_lookup),
                        ("best/7ep (NB11)", best_7ep_lookup),
                        ("best/20ep (NB12)", lookup)]:
        per_class_all_seeds = [
            lkp[event][(HEATMAP_BUDGET, s)]["test_per_class_f1"]
            for s in SEED_SETS
            if (HEATMAP_BUDGET, s) in lkp[event]
            and "test_per_class_f1" in lkp[event][(HEATMAP_BUDGET, s)]
        ]
        if per_class_all_seeds:
            mean_per_class = [
                statistics.mean(seed[i] for seed in per_class_all_seeds)
                for i in range(len(per_class_all_seeds[0]))
            ]
            data_rows.append(mean_per_class)
            row_labels.append(label)

    if len(data_rows) < 2:
        print(f"  Skipping {event}: insufficient data for comparison.")
        continue

    data = np.array(data_rows)
    n_classes = data.shape[1]
    class_labels = CLASS_LABELS[:n_classes] if n_classes <= len(CLASS_LABELS) else [
        f"class_{i}" for i in range(n_classes)
    ]

    fig, ax = plt.subplots(
        figsize=(max(9, n_classes * 0.75), len(row_labels) * 0.65 + 1.8)
    )
    im = ax.imshow(data, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)

    ax.set_xticks(range(n_classes))
    ax.set_xticklabels(class_labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=9)
    ax.set_title(
        f"{event}  |  Budget={HEATMAP_BUDGET}  |  Per-class F1 (mean across seeds)",
        fontsize=10,
    )

    for i in range(len(row_labels)):
        for j in range(n_classes):
            val = data[i, j]
            color = "black" if 0.25 < val < 0.75 else "white"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7, color=color)

    fig.colorbar(im, ax=ax, label="F1 Score")
    plt.tight_layout()
    plt.show()

In [ ]:
# Phase 1 Best Epoch Analysis
# With 20 epochs, the distribution of best epochs is particularly informative:
# if best epochs cluster around 3-7, Phase 1 models overfit quickly and the
# paper's default of 7 epochs was already near-optimal. If best epochs spread
# across 7-15, the extra epochs provide genuine benefit.

best_epochs = []  # list of (event, budget, seed_set, best_epoch)
best_epochs_by_budget = {b: [] for b in BUDGETS}

for event in TARGET_EVENTS:
    for budget in BUDGETS:
        for seed_set in SEED_SETS:
            r = lookup[event].get((budget, seed_set))
            if r and r.get("phase1_best_epoch") is not None:
                ep = r["phase1_best_epoch"]
                best_epochs.append((event, budget, seed_set, ep))
                best_epochs_by_budget[budget].append(ep)

print(f"Phase 1 Best Epoch Distribution ({len(best_epochs)} experiments, max={WEIGHT_GEN_EPOCHS})")
print("=" * 60)

if best_epochs:
    all_ep_vals = [x[3] for x in best_epochs]
    print(f"Overall: mean={statistics.mean(all_ep_vals):.2f}, "
          f"median={statistics.median(all_ep_vals):.1f}, "
          f"min={min(all_ep_vals)}, max={max(all_ep_vals)}")
    print()

    # Per-budget breakdown
    print(f"{'Budget':<8} {'Mean':>6} {'Median':>8} {'Min':>5} {'Max':>5} {'n':>4}")
    print("-" * 40)
    for budget in BUDGETS:
        eps = best_epochs_by_budget[budget]
        if eps:
            print(f"{budget:<8} {statistics.mean(eps):>6.2f} {statistics.median(eps):>8.1f} "
                  f"{min(eps):>5} {max(eps):>5} {len(eps):>4}")

    # Histogram
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    # Overall histogram
    ax = axes[0]
    bins = np.arange(0.5, WEIGHT_GEN_EPOCHS + 1.5, 1)
    ax.hist(all_ep_vals, bins=bins, color="darkorange", alpha=0.85, edgecolor="black")
    ax.axvline(x=7, color="steelblue", linestyle="--", linewidth=1.5, label="Paper default (7)")
    ax.set_xlabel("Best Phase 1 Epoch (1-indexed)")
    ax.set_ylabel("Count")
    ax.set_title(f"Overall Distribution of Best Phase 1 Epoch (max={WEIGHT_GEN_EPOCHS})")
    ax.set_xticks(range(1, WEIGHT_GEN_EPOCHS + 1))
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

    # Per-budget histogram (stacked)
    ax = axes[1]
    budget_data = [best_epochs_by_budget[b] for b in BUDGETS]
    ax.hist(budget_data, bins=bins, label=[f"Budget={b}" for b in BUDGETS],
            alpha=0.85, edgecolor="black", stacked=True)
    ax.axvline(x=7, color="steelblue", linestyle="--", linewidth=1.5, label="Paper default (7)")
    ax.set_xlabel("Best Phase 1 Epoch (1-indexed)")
    ax.set_ylabel("Count")
    ax.set_title("Best Phase 1 Epoch by Budget")
    ax.set_xticks(range(1, WEIGHT_GEN_EPOCHS + 1))
    ax.legend(fontsize=8)
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

    # How often is best epoch > 7 (benefiting from extra epochs)?
    n_above_7 = sum(1 for ep in all_ep_vals if ep > 7)
    n_at_or_below_7 = sum(1 for ep in all_ep_vals if ep <= 7)
    n_at_last = sum(1 for ep in all_ep_vals if ep == WEIGHT_GEN_EPOCHS)
    pct_above_7 = 100 * n_above_7 / len(all_ep_vals)
    pct_at_last = 100 * n_at_last / len(all_ep_vals)
    print(f"\nBest epoch <= 7 (paper's range): {n_at_or_below_7}/{len(all_ep_vals)} ({100-pct_above_7:.1f}%)")
    print(f"Best epoch > 7  (extra epochs):   {n_above_7}/{len(all_ep_vals)} ({pct_above_7:.1f}%)")
    print(f"Best epoch = {WEIGHT_GEN_EPOCHS} (last epoch):     {n_at_last}/{len(all_ep_vals)} ({pct_at_last:.1f}%)")
    print()
    if pct_above_7 > 30:
        print("A significant fraction benefits from epochs beyond 7 -- the extra epochs help.")
    elif pct_above_7 > 10:
        print("Some experiments benefit from extra epochs, but most peak within the paper's 7.")
    else:
        print("Very few experiments benefit from epochs beyond 7 -- the paper's default was sufficient.")
else:
    print("No phase1_best_epoch data found. Run experiments first.")

In [ ]:
# Rebuild multi-tab dashboard with all strategy result sets.

from lg_cotrain.dashboard import discover_result_sets, generate_html_multi

TOP_RESULTS_ROOT = str(repo_root / "results")

result_sets = discover_result_sets(TOP_RESULTS_ROOT)
print(f"Discovered {len(result_sets)} model(s):")
for model, types in result_sets.items():
    for exp_type, experiments in types.items():
        for name, path in experiments:
            print(f"  {model}/{exp_type}/{name:<20} -> {path}")

html = generate_html_multi(result_sets, data_root=DATA_ROOT)
dashboard_path = Path(TOP_RESULTS_ROOT) / "dashboard.html"
dashboard_path.write_text(html)
print(f"\nDashboard written to: {dashboard_path}")
print("Open in a browser to compare all strategies across all budgets and events.")